In [ ]:
from data import *
from metrics import *
from train import *
from torch.optim import AdamW

In [ ]:
from torch import nn
class MyDEMUCS(nn.Module):

  class encoder_block(nn.Module):
    def __init__(self, c_in, c_out, K, S):
      super().__init__()
      self.K = K
      self.fs = nn.Conv1d(c_in, c_out, K, S, 0)
      self.sc = nn.Conv1d(c_out, 2*c_out, 1, 1, 0)
      self.glu = nn.GLU(dim=-2)

    def forward(self, x):
      x = nn.functional.pad(x, (self.K-1, 0))
      x = nn.functional.relu(self.fs(x))
      x = self.glu(self.sc(x))
      return x

  class decoder_block(nn.Module):
    def __init__(self, c_in, c_out, K, S):
      super().__init__()
      self.pd = K - S
      self.fs = nn.Conv1d(c_in, c_in * 2, 1, 1, 0)
      self.sc = nn.ConvTranspose1d(c_in, c_out, K, S, 0)
      self.glu = nn.GLU(dim=-2)

    def forward(self, x):
      x = self.glu(self.fs(x))
      x = self.sc(x)
      if self.pd > 0:
        x = x[:,:,:-self.pd]
      return x

  def __init__(self, L, H, K, S, U=1):
    super().__init__()
    self.shr = S ** L

    self.bottleneck = nn.LSTM(input_size = H << L-1, hidden_size = H << L-1,
                              num_layers=2, batch_first = True)

    enc = {'encoder_1': MyDEMUCS.encoder_block(1, H, K, S)}
    dec = {'decoder_1': MyDEMUCS.decoder_block(H, 1, K, S)}
    for i in range(2, L+1):
      ch = H << i-2
      enc[f'encoder_{i}'] = MyDEMUCS.encoder_block(ch, ch * 2, K, S)
      dec[f'decoder_{i}'] = MyDEMUCS.decoder_block(2 * ch, ch, K, S)
    self.encoder = nn.ModuleDict(enc)
    self.decoder = nn.ModuleDict(dec)

  def forward(self, x):
    l = x.shape[-1]
    if l % self.shr:
      padd_l = self.shr - l % self.shr
      #print(padd_l)
      x = nn.functional.pad(x, (0, padd_l))
    enc_outputs = []

    Layers = sorted(self.encoder.items(), key = lambda x : int(x[0][8:]))
    for name, layer in Layers:
      x = layer(x)
      enc_outputs.append(x)

    x = torch.permute(x, (0, 2, 1))
    x, _ = self.bottleneck(x)
    x = torch.permute(x, (0, 2, 1))

    Layers = sorted(self.decoder.items(), key = lambda x : int(x[0][8:]))
    for name, layer in Layers[::-1]:
      i = int(name[8:])
      x = layer(x + enc_outputs[i - 1])
    return x

In [ ]:
model = MyDEMUCS(5, 10, 6, 3)

In [ ]:
report = train_a_model(model,
                        nn.L1Loss(),
                        AdamW(model.parameters(), lr=3e-5),
                        train_data,
                        shortened_train_data,
                        evaluation_data,
                        '/content/drive/MyDrive/DEMUCS_runs',
                        epochs = 20)